# Q3 — Audio Entropy Analysis (Normalized)
#
# Pipeline:
#   (1) Load raw audio at 16 kHz mono
#   (2) Remove DC offset
#   (3) Temporal trim: skip first 0.5 s, then take 4 s → all clips are exactly 4 s
#   (4) Peak-normalize to [-1, 1]
#   (5) Compute entropy metrics

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
import numpy as np
import pandas as pd
import librosa
import os

PATH_AMBIENT = '/content/drive/MyDrive/Learning-document/ProbStat/data/Q3_Ambient'
PATH_MUSIC   = '/content/drive/MyDrive/Learning-document/ProbStat/data/Q3_Music'

TARGET_SR = 16000
CLIP_DURATION = 4.0  # seconds
CLIP_SAMPLES = int(TARGET_SR * CLIP_DURATION)  # 64000 samples
TRIM_SECONDS = 0.5

def load_clip(path, sr=TARGET_SR, trim_seconds=TRIM_SECONDS, duration=CLIP_SAMPLES):
    """Load audio, skip first 0.5s, take 4s, remove DC, peak-normalize to [-1,1]."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    # Skip first 0.5s (consistent with temporal trimming in preprocessing)
    trim_samples = int(trim_seconds * sr)
    audio = audio[trim_samples:]
    # Take first 4s
    audio = audio[:duration]
    # Remove DC offset
    audio = audio - np.mean(audio)
    # Peak normalize
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

# Verify paths
for label, path in [('Ambient', PATH_AMBIENT), ('Music', PATH_MUSIC)]:
    files = sorted([f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))])
    print(f'{label}: {len(files)} files in {path}')

Ambient: 10 files in /content/drive/MyDrive/Learning-document/ProbStat/data/Q3_Ambient
Music: 10 files in /content/drive/MyDrive/Learning-document/ProbStat/data/Q3_Music


In [41]:
# Load all clips and concatenate for global bin width
all_audio = []
clip_data = {}  # {class: [audio_array, ...]}

for label, path in [('Ambient', PATH_AMBIENT), ('Music', PATH_MUSIC)]:
    files = sorted([f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))])
    clips = []
    for f in files:
        audio = load_clip(os.path.join(path, f))
        clips.append(audio)
        all_audio.append(audio)
    clip_data[label] = clips
    print(f'{label}: loaded {len(clips)} clips, each {CLIP_SAMPLES} samples')

all_audio = np.concatenate(all_audio)
n_total = len(all_audio)

# Freedman-Diaconis rule
q75, q25 = np.percentile(all_audio, [75, 25])
iqr = q75 - q25
delta = 2 * iqr / n_total ** (1/3)
K = int(np.ceil((2.0) / delta))  # bins over [-1, 1]
H_max = np.log2(K)

print(f'\nTotal samples: {n_total:,}')
print(f'IQR: {iqr:.6f}')
print(f'Bin width (delta): {delta:.6f}')
print(f'Number of bins K: {K}')
print(f'H_max = log2(K): {H_max:.4f}')

Ambient: loaded 10 clips, each 64000 samples
Music: loaded 10 clips, each 64000 samples

Total samples: 1,280,000
IQR: 0.200423
Bin width (delta): 0.003692
Number of bins K: 542
H_max = log2(K): 9.0821


In [42]:
from scipy.stats import shapiro

bin_edges = np.arange(-1, 1 + delta, delta)

results = []
for label in ['Ambient', 'Music']:
    for i, audio in enumerate(clip_data[label], 1):
        counts, _ = np.histogram(audio, bins=bin_edges)
        p = counts / counts.sum()
        p = p[p > 0]

        H_Q = -np.sum(p * np.log2(p))
        h_hat = H_Q + np.log2(delta)
        sigma2 = np.var(audio)
        H_gauss = 0.5 * np.log2(2 * np.pi * np.e * sigma2) if sigma2 > 0 else 0
        gap = H_gauss - h_hat

        # Shapiro-Wilk W (subsample to 5000 for scipy limit)
        rng = np.random.default_rng(42)
        sub = rng.choice(audio, size=5000, replace=False)
        W, _ = shapiro(sub)

        results.append({
            'Class': label,
            'Clip': i,
            'H_Q': round(H_Q, 3),
            'h_hat': round(h_hat, 3),
            'H_max': round(H_max, 3),
            'H_gauss': round(H_gauss, 3),
            'gap': round(gap, 3),
            'W': round(W, 4),
        })

df = pd.DataFrame(results)

# === Two separate tables ===
for label in ['Ambient', 'Music']:
    sub = df[df['Class'] == label][['Clip', 'H_Q', 'h_hat', 'H_max', 'H_gauss', 'gap', 'W']].copy()
    sub = sub.reset_index(drop=True)
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(sub.to_string(index=False))


  Ambient
 Clip   H_Q  h_hat  H_max  H_gauss   gap      W
    1 7.193 -0.889  9.082   -0.838 0.051 0.9774
    2 6.756 -1.325  9.082   -1.252 0.073 0.9507
    3 7.903 -0.179  9.082   -0.173 0.006 0.9996
    4 7.960 -0.121  9.082   -0.031 0.090 0.9997
    5 8.117  0.035  9.082    0.119 0.084 0.9997
    6 7.829 -0.252  9.082   -0.157 0.095 0.9989
    7 8.399  0.317  9.082    0.342 0.024 0.9939
    8 7.934 -0.148  9.082   -0.113 0.035 0.9991
    9 8.044 -0.037  9.082   -0.009 0.028 0.9984
   10 7.594 -0.488  9.082   -0.473 0.015 0.9947

  Music
 Clip   H_Q  h_hat  H_max  H_gauss   gap      W
    1 7.206 -0.875  9.082   -0.756 0.119 0.9632
    2 6.484 -1.597  9.082   -1.441 0.156 0.9379
    3 7.587 -0.494  9.082   -0.459 0.035 0.9891
    4 7.416 -0.665  9.082   -0.637 0.028 0.9896
    5 7.208 -0.873  9.082   -0.835 0.039 0.9898
    6 7.475 -0.606  9.082   -0.533 0.073 0.9712
    7 7.005 -1.076  9.082   -0.830 0.246 0.9326
    8 7.010 -1.072  9.082   -0.897 0.175 0.9347
    9 7.387 -0.694  

In [43]:
# Summary statistics per class
summary = df.groupby('Class')[['H_Q', 'h_hat', 'H_gauss', 'gap', 'W']].agg(['mean', 'std'])
print('=== Summary Statistics ===')
display(summary.round(3))

=== Summary Statistics ===


H_Q         h_hat        H_gauss           gap             W       
          mean    std   mean    std    mean    std   mean    std   mean    std
Class                                                                         
Ambient  7.773  0.479 -0.309  0.479  -0.258  0.474  0.050  0.033  0.991  0.016
Music    7.257  0.368 -0.824  0.368  -0.724  0.317  0.099  0.077  0.966  0.025